# Implied Volatility & the Volatility Surface

From-scratch implementation of implied volatility computation via Newton-Raphson and bisection,
the volatility smile/skew, volatility surface construction, and a simplified VIX calculation.

---

## 1. What Is Implied Volatility, and Why Does It Matter?

### The Big Picture

The Black-Scholes-Merton model takes five inputs --- stock price ($S$), strike ($K$), risk-free rate ($r$), time to expiry ($T$), and volatility ($\sigma$) --- and produces an option price. Four of these inputs are directly observable in the market. But **volatility is not**.

This creates a natural question: if we know the market price of an option, can we work *backwards* to find the volatility that the market is "implying"?

**Implied volatility (IV)** is the answer: it is the value of $\sigma$ that, when plugged into the BSM formula, reproduces the observed market price.

### The Insurance Analogy

Think of an insurance company that prices policies using a model. If you see the price of a flood insurance policy, you can work backwards to figure out what probability of flooding the company is assuming. The "implied flood probability" tells you what the market thinks about flood risk --- even if the company never explicitly states it.

Similarly, implied volatility tells you what the options market thinks about future stock volatility --- the market's collective "fear gauge."

> **Key Concept:** Implied volatility is not a prediction of future volatility. It is the market's consensus about uncertainty, as expressed through option prices. When IV is high, the market is nervous. When IV is low, the market is calm.

### IV as the Market's Fear Gauge

To understand why IV functions as a fear gauge, think about what happens during a market crisis. Imagine a stock trading at \$100 during normal times versus during a financial panic:

- **Normal times:** Traders expect the stock to fluctuate maybe 15--20% per year. Options reflect this calm outlook --- they are relatively cheap, and IV hovers around 15--20%.
- **During a crisis:** Nobody knows what will happen tomorrow. The stock might crash 30% or bounce 20%. Traders bid up option prices because options provide protection. As option prices rise, the IV extracted from those prices also rises --- perhaps to 50% or even 80%.

The IV did not rise because anyone sat down and calculated that future realized volatility would be 50%. It rose because *fear itself* drove option prices higher. People were willing to pay more for insurance, and IV is the thermometer reading of that willingness.

> **Important:** There is a subtle but critical asymmetry. IV tends to spike sharply when markets fall ("the elevator down") but decline gradually when markets rise ("the escalator up"). This asymmetry --- known as the **leverage effect** --- means that implied volatility and stock prices are negatively correlated. Bad news moves IV more than good news of equal magnitude.

### A Concrete Example: What IV = 30% Actually Means

When a trader says "this option has 30 vol," they mean $\sigma_{\text{imp}} = 0.30$, or 30% annualized volatility. Under the BSM assumptions (which assume log-normal returns), this implies:

- **Over one year**, the stock is expected (by the market) to move within roughly $\pm 30\%$ of its current price about 68% of the time (one standard deviation).
- **Over one month**, the expected range is $\pm 30\% \times \sqrt{1/12} \approx \pm 8.7\%$.
- **Over one day**, the expected range is $\pm 30\% \times \sqrt{1/252} \approx \pm 1.9\%$.

So a stock at \$100 with IV = 30% is expected to have daily moves of roughly \$1.90. If IV rises to 60%, the market is now expecting daily moves of roughly \$3.80 --- double the uncertainty.

> **Common Mistake:** Beginners often confuse IV with a directional forecast. IV = 30% does **not** mean the stock will go up or down 30%. It means the market expects the stock to fluctuate with an annualized standard deviation of 30%. It says nothing about direction --- only about the magnitude of expected moves.

### Why Traders Care

IV serves three critical purposes:

| Purpose | Explanation |
|---------|-------------|
| **Quoting convention** | Traders quote options in IV rather than price. Saying "the 100 call is at 22 vol" is more informative than saying "it costs \$8.95" because IV is comparable across different strikes and expirations. |
| **Relative value** | If two similar options have different IVs, the higher one is "more expensive." This helps identify mispricings. |
| **Market sentiment** | Rising IV signals increasing fear or uncertainty. The VIX (the "fear index") is built from implied volatilities. |

The quoting convention point deserves emphasis. Consider two call options:

- Option A: Strike \$100, 30 days to expiry, price \$3.50
- Option B: Strike \$110, 90 days to expiry, price \$4.20

Which is "more expensive"? The dollar prices are nearly useless for comparison because the options differ in both strike and maturity. But if we compute IV for each --- say 22% for A and 28% for B --- we can immediately see that B is relatively more expensive. The market is assigning more uncertainty to that strike/maturity combination.

> **Important:** If BSM were a perfect model, implied volatility would be the same for all strikes and expirations of the same underlying. In reality, it is not --- the patterns of IV across strikes (the "smile" or "skew") and across maturities (the "term structure") reveal the limitations of BSM and the market's true risk assessment.

---

## 2. The Mathematical Setup

### The Inversion Problem

Given a market price $V_{\text{mkt}}$ for a European option, the **implied volatility** $\sigma_{\text{imp}}$ is the unique $\sigma > 0$ satisfying:

$$V_{\text{BSM}}(S, K, r, T, \sigma_{\text{imp}}) = V_{\text{mkt}}$$

In words: "Find the volatility that makes BSM agree with the market."

This is a **root-finding problem**: we are looking for the root of $f(\sigma) = V_{\text{BSM}}(\sigma) - V_{\text{mkt}} = 0$.

### Why We Solve Backwards (Price to Vol) Instead of Forwards

You might wonder: why not just measure volatility directly from historical data and use *that* to price options? The reason is that historical (or "realized") volatility tells you what *already happened*. Implied volatility tells you what the market *expects to happen*.

Consider an analogy. Suppose you want to know the probability that it will rain tomorrow:

- **Historical approach:** Look at the last 30 days. It rained on 10 of them, so estimate 33%. This ignores the fact that a massive storm system is approaching tonight.
- **Market-implied approach:** Look at the price of "rain insurance" (umbrellas, event cancellation policies). If insurance is expensive, the market believes rain is likely. This captures *forward-looking* information that history cannot.

Implied volatility works the same way. A company might have had calm 15% historical volatility, but if an earnings announcement is tomorrow, options will be expensive and IV might be 50%. The market knows something is coming, even though history looks calm.

> **Key Concept:** The "backwards" nature of IV --- solving from observed prices to the unobservable parameter --- is actually a feature, not a bug. It extracts the market's forward-looking risk assessment, which incorporates all available information (news, sentiment, positioning, insider knowledge) far more efficiently than any backward-looking statistical estimate could.

### Why a Unique Solution Exists

Three facts guarantee that we can always find a unique IV:

1. **Vega is always positive**: $\frac{\partial V}{\partial \sigma} = S\sqrt{T}\,n(d_1) > 0$ for all $\sigma > 0$. This means the BSM price is **strictly increasing** in $\sigma$. Higher volatility always means higher option price.

2. **Lower bound**: As $\sigma \to 0^+$, $C \to \max(S - Ke^{-rT}, 0)$ (the intrinsic value).

3. **Upper bound**: As $\sigma \to \infty$, $C \to S$ (for calls) and $P \to Ke^{-rT}$ (for puts).

Since the BSM price is continuous, strictly increasing, and goes from intrinsic value to $S$, by the intermediate value theorem, any market price in this range corresponds to exactly one $\sigma$.

Let us build intuition for why these bounds make sense:

- **When $\sigma \to 0$:** Zero volatility means the stock price is deterministic. You know exactly where it will end up. A call option is worth exactly its discounted payoff at expiry --- either $S - Ke^{-rT}$ if in the money, or zero. There is no "optionality" because there is no uncertainty.
- **When $\sigma \to \infty$:** Infinite volatility means the stock can go anywhere. A call option is almost certain to finish in the money (the stock will eventually drift to arbitrarily high values), so it approaches the value of the stock itself. The strike becomes irrelevant compared to the enormous possible movements.

Between these extremes, vega being positive means the price smoothly and monotonically increases. This is the mathematical guarantee that IV is well-defined.

> **Key Concept:** The monotonicity of option prices in volatility is what makes IV well-defined. If option prices could decrease with volatility, there might be multiple values of $\sigma$ giving the same price, and IV would be ambiguous.

> **Common Mistake:** Trying to compute IV for an option price below the intrinsic value or above the stock price. No valid IV exists in these cases --- the option price is outside the BSM feasible range, indicating an arbitrage opportunity or stale data.

### Setting Up the Code

Before we implement the IV solvers, we need the BSM pricing functions and a visualization of the price-vs-volatility relationship. The code below defines:

- `bsm_d1_d2`: computes the $d_1$ and $d_2$ terms from the BSM formula.
- `bsm_call` / `bsm_put`: the closed-form BSM prices.
- `bsm_vega`: the sensitivity of the option price to volatility, $\partial V / \partial \sigma = S\sqrt{T}\,n(d_1)$.

We then plot BSM call price and vega as functions of $\sigma$ to visually confirm the monotonicity that guarantees a unique IV solution.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

In [ ]:
# BSM pricing functions
def bsm_d1_d2(S, K, r, T, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return d1, d2

def bsm_call(S, K, r, T, sigma):
    d1, d2 = bsm_d1_d2(S, K, r, T, sigma)
    return S * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)

def bsm_put(S, K, r, T, sigma):
    d1, d2 = bsm_d1_d2(S, K, r, T, sigma)
    return K * np.exp(-r * T) * stats.norm.cdf(-d2) - S * stats.norm.cdf(-d1)

def bsm_vega(S, K, r, T, sigma):
    d1, _ = bsm_d1_d2(S, K, r, T, sigma)
    return S * np.sqrt(T) * stats.norm.pdf(d1)

# Show BSM price as function of sigma
S0, K, r, T = 100, 100, 0.05, 0.5
sigma_range = np.linspace(0.01, 1.0, 200)
prices = np.array([bsm_call(S0, K, r, T, s) for s in sigma_range])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sigma_range, prices, color=PRIMARY, linewidth=2)
axes[0].set_xlabel(r'$\sigma$')
axes[0].set_ylabel('Call Price')
axes[0].set_title(r'BSM Call Price vs $\sigma$ (monotonically increasing)')

vegas = np.array([bsm_vega(S0, K, r, T, s) for s in sigma_range])
axes[1].plot(sigma_range, vegas, color=SECONDARY, linewidth=2)
axes[1].set_xlabel(r'$\sigma$')
axes[1].set_ylabel('Vega')
axes[1].set_title(r'Vega = $\partial V / \partial \sigma > 0$')

plt.tight_layout()
plt.show()

### Interpreting the Plots

The left plot confirms that the call price increases monotonically with volatility. Given any market price (imagine drawing a horizontal line), there is exactly one intersection --- that is the implied volatility.

The right plot shows that vega (the derivative of price with respect to volatility) is always positive, confirming the monotonicity. Vega is largest for at-the-money options at moderate volatilities.

Notice the shape of the vega curve: it peaks around $\sigma \approx 0.10$--$0.20$ for this ATM option, then gradually declines. This has practical implications:

- **Near the peak:** A small change in $\sigma$ produces a large change in price. Newton-Raphson works beautifully here because vega (the denominator) is large and well-behaved.
- **In the tails:** Vega is small. This means price is relatively insensitive to volatility, making the IV extraction problem harder --- small price differences correspond to large IV differences.

> **Key Concept:** Vega measures how much an option's price changes per unit change in volatility. ATM options have the highest vega because they sit on the "knife's edge" of expiring in or out of the money --- volatility has the most influence on their outcome. Deep ITM and deep OTM options have low vega because their outcome is already largely determined regardless of volatility.

> **Common Mistake:** Assuming vega is constant. In reality, vega itself changes as the underlying moves, as volatility changes, and as time passes. This second-order effect (the change of vega with respect to volatility) is called **vomma** or **volga**, and it matters for managing large option portfolios.

---

## 3. Newton-Raphson Method: Smart Guessing Using Vega

### The Core Idea

Newton-Raphson is the workhorse algorithm for finding implied volatility. The idea is simple: start with a guess for $\sigma$, compute the price error, and use the derivative (vega) to make a better guess.

Think of it like this: you are standing on a hillside and want to find the point where the ground is level (the root). You can see the slope beneath your feet. If the ground slopes steeply downhill to the right, the level point is probably not far to the right. If it slopes gently, the level point might be much further away. Newton-Raphson uses this slope information to make intelligent leaps.

### The Algorithm

Given a current guess $\sigma_n$:

$$\sigma_{n+1} = \sigma_n - \frac{V_{\text{BSM}}(\sigma_n) - V_{\text{mkt}}}{\text{Vega}(\sigma_n)}$$

In words: "Adjust the guess by the price error divided by the sensitivity."

Let us break this formula into its components to build deep intuition:

- **Numerator** $V_{\text{BSM}}(\sigma_n) - V_{\text{mkt}}$: This is the **price error** --- how far off our current guess is. If positive, our guess produces a price that is too high, meaning our $\sigma$ is too large.
- **Denominator** $\text{Vega}(\sigma_n)$: This is the **sensitivity** --- how much the BSM price changes per unit change in $\sigma$. It tells us the "exchange rate" between price errors and volatility errors.
- **The ratio**: Dividing the price error by vega converts the error from "price units" to "volatility units." If the price is \$2 too high and vega is \$20 per vol point, we need to reduce $\sigma$ by $2/20 = 0.10$.

> **Key Concept:** Newton-Raphson is essentially asking: "Given how sensitive the price is to volatility *right here*, how much should I adjust my volatility guess to eliminate the current price error?" It is like a thermostat that knows the relationship between the dial setting and the temperature --- it can jump directly to the right setting instead of blindly turning the dial.

### Worked Example: Hand-Computing IV Step by Step

Let us trace through a complete Newton-Raphson computation by hand. Suppose we have an ATM call option with:
- $S = 100$, $K = 100$, $r = 0.05$, $T = 0.5$
- Market price $V_{\text{mkt}} = \$8.00$
- Starting guess $\sigma_0 = 0.50$

**Iteration 1:**
- Compute BSM price at $\sigma = 0.50$: $V_{\text{BSM}}(0.50) = \$15.22$ (way too high!)
- Compute vega at $\sigma = 0.50$: $\text{Vega} = 18.74$
- Price error: $15.22 - 8.00 = 7.22$
- Newton step: $\sigma_1 = 0.50 - 7.22 / 18.74 = 0.50 - 0.385 = 0.115$

**Iteration 2:**
- Compute BSM price at $\sigma = 0.115$: $V_{\text{BSM}}(0.115) \approx \$5.75$ (now too low)
- Compute vega at $\sigma = 0.115$: $\text{Vega} \approx 27.4$
- Price error: $5.75 - 8.00 = -2.25$
- Newton step: $\sigma_2 = 0.115 - (-2.25) / 27.4 = 0.115 + 0.082 = 0.197$

**Iteration 3:**
- We are now close. A few more iterations will refine to machine precision.

Notice how the algorithm oscillates and converges: first overshooting low, then correcting. By iteration 3 we are already in the right neighborhood, and quadratic convergence takes over.

### Why Vega Is the Key

The reason Newton-Raphson works so well for IV is that we have an analytical expression for vega. In many root-finding problems, you need to approximate the derivative numerically (which is slow and noisy). Here, the BSM formula gives us the exact derivative:

$$\text{Vega} = S\sqrt{T}\,\phi(d_1)$$

where $\phi$ is the standard normal PDF. This formula is cheap to evaluate and exact, making Newton-Raphson particularly efficient for this problem.

### Convergence Properties

| Property | Newton-Raphson | Bisection |
|----------|---------------|-----------|
| **Convergence rate** | Quadratic ($O(\epsilon^2)$) | Linear ($O(\epsilon)$) |
| **Iterations (typical)** | 3--6 | 30--50 |
| **Requires derivative** | Yes (vega) | No |
| **Guaranteed convergence** | No (can fail for extreme cases) | Yes (always converges) |
| **Best for** | ATM and near-ATM options | Deep OTM/ITM options |

> **Key Concept:** "Quadratic convergence" means the number of correct digits roughly doubles at each step. If you have 2 correct digits, the next iteration gives 4, then 8, then 16. This is why Newton-Raphson typically converges in just 3--6 iterations. Compare this to bisection, which adds only about 0.3 decimal digits per iteration.

### When Newton-Raphson Fails

Despite its speed, Newton-Raphson can fail in specific scenarios:

1. **Deep OTM options:** Vega is extremely small (the option price barely responds to volatility changes). Dividing by a tiny vega produces enormous, unstable jumps.
2. **Near-zero time to expiry:** Very short-dated options have razor-thin vega except right at the money. Newton-Raphson can overshoot wildly.
3. **Bad initial guess:** If the starting $\sigma_0$ is far from the true IV and the BSM price-vs-sigma curve has strong curvature in that region, the linear approximation can send us to negative territory.

> **Common Mistake:** Newton-Raphson can fail when vega is very small (deep OTM/ITM options). In this case, you are dividing by a tiny number, which creates huge jumps. Always include a fallback to bisection for robustness.

### Implementation and Verification

The code below implements Newton-Raphson for IV extraction. Key implementation details:

- We track the full iteration history `(sigma, |error|)` so we can visualize convergence.
- The `max(sigma, 1e-6)` guard prevents $\sigma$ from going negative (which would be meaningless).
- The vega floor check `abs(vega) < 1e-15` prevents division by zero --- if vega is essentially zero, we bail out rather than producing garbage.

To verify correctness, we generate a "market price" from a known $\sigma = 0.25$, then run the solver and check that it recovers the true value. The convergence history demonstrates the quadratic convergence pattern.

In [ ]:
def implied_vol_newton(market_price, S, K, r, T, option_type='call',
                        sigma0=0.3, tol=1e-10, max_iter=100):
    """Compute implied volatility via Newton-Raphson.
    
    Returns
    -------
    sigma : float -- implied volatility
    iterations : int
    history : list of (sigma, error) tuples
    """
    sigma = sigma0
    history = []
    
    price_fn = bsm_call if option_type == 'call' else bsm_put
    
    for i in range(max_iter):
        price = price_fn(S, K, r, T, sigma)
        vega = bsm_vega(S, K, r, T, sigma)
        error = price - market_price
        history.append((sigma, abs(error)))
        
        if abs(error) < tol:
            return sigma, i + 1, history
        
        if abs(vega) < 1e-15:
            break
        
        sigma = sigma - error / vega
        sigma = max(sigma, 1e-6)  # keep positive
    
    return sigma, max_iter, history


# Test: find IV for a known price
true_sigma = 0.25
market_price = bsm_call(S0, K, r, T, true_sigma)

iv, iters, history = implied_vol_newton(market_price, S0, K, r, T, 'call', sigma0=0.5)

print(f"True sigma:     {true_sigma:.10f}")
print(f"Found IV:       {iv:.10f}")
print(f"Iterations:     {iters}")
print(f"Error:          {abs(iv - true_sigma):.2e}")

# Convergence history
print(f"\n{'Iter':>4s} {'sigma':>14s} {'|error|':>14s}")
for i, (s, e) in enumerate(history):
    print(f"{i+1:4d} {s:14.10f} {e:14.2e}")

### Reading the Convergence Output

Newton-Raphson converges in just a few iterations, with the error dropping dramatically at each step. This is the hallmark of quadratic convergence --- the number of correct digits roughly doubles each iteration.

Look carefully at the `|error|` column. You should see a pattern like:

- Iteration 1: error $\sim 10^{0}$ (way off)
- Iteration 2: error $\sim 10^{-1}$ (getting closer)
- Iteration 3: error $\sim 10^{-3}$ (quadratic kick-in)
- Iteration 4: error $\sim 10^{-7}$ (nearly there)
- Iteration 5: error $\sim 10^{-14}$ (machine precision)

The jump from $10^{-3}$ to $10^{-7}$ to $10^{-14}$ is the signature of quadratic convergence: each step roughly *squares* the error. This is why Newton-Raphson is so fast once it enters the "convergence basin" near the true root.

> **Important:** The first 1--2 iterations may not show quadratic convergence because the initial guess is far from the root (we started at $\sigma_0 = 0.50$ when the true answer is $0.25$). Quadratic convergence is a *local* property --- it kicks in once you are "close enough." This is another reason to choose a reasonable starting guess.

---

## 4. Bisection Method: Guaranteed but Slower

### When Newton-Raphson Is Not Enough

For deep out-of-the-money options, vega is tiny and Newton-Raphson can become unstable. The **bisection method** is the reliable fallback: slower, but guaranteed to converge.

### The Core Idea

Bisection is like a binary search. We know the IV lies between some lower bound $\sigma_{\text{lo}}$ and upper bound $\sigma_{\text{hi}}$. At each step:

1. Try the midpoint: $\sigma_{\text{mid}} = (\sigma_{\text{lo}} + \sigma_{\text{hi}})/2$
2. If BSM($\sigma_{\text{mid}}$) is too high, the answer is in the lower half.
3. If it is too low, the answer is in the upper half.
4. Repeat.

Each iteration halves the interval, so after $n$ iterations, the uncertainty is reduced by $2^n$.

### An Everyday Analogy

Bisection is the algorithm you use when guessing a number between 1 and 100:

- "Is it above 50?" Yes.
- "Is it above 75?" No.
- "Is it above 62?" Yes.
- "Is it above 68?" No.
- ...

Each question eliminates half the remaining possibilities. After 7 questions, you have narrowed 100 possibilities to about 1. After 34 questions, you have narrowed a range of $[0.001, 5.0]$ to machine precision ($\sim 10^{-10}$).

The crucial guarantee: **bisection always works**, as long as the function changes sign across the interval. Since BSM price is monotonic in $\sigma$, if our initial bracket contains the true IV, bisection will find it.

### When to Use Each Method

In practice, many production systems use a **hybrid** approach:

1. Start with Newton-Raphson (fast for the common case).
2. If Newton-Raphson produces a negative $\sigma$, exceeds the bracket, or fails to converge in ~10 iterations, switch to bisection.
3. Some systems use one Newton-Raphson step to refine the bisection midpoint, combining the best of both worlds.

This hybrid strategy gives you Newton's speed for 95% of cases and bisection's reliability for the remaining 5%.

> **Key Concept:** Bisection converges linearly --- each iteration adds about one binary digit of accuracy ($\log_2 10 \approx 3.3$ iterations per decimal digit). It needs about 30--50 iterations for full double-precision accuracy, compared to 3--6 for Newton-Raphson. The trade-off: slower but bulletproof.

> **Common Mistake:** Setting the initial bracket too tight. If $\sigma_{\text{hi}}$ is not large enough to produce a BSM price above the market price, bisection will fail to converge. Always use a generous upper bound (e.g., $\sigma_{\text{hi}} = 5.0$) to ensure the bracket contains the true IV.

### Implementation and Convergence Comparison

The code below implements bisection and then runs both solvers on the same problem so we can compare convergence rates side by side. Watch the log-scale error plot --- the visual difference between quadratic and linear convergence is striking.

In [ ]:
def implied_vol_bisection(market_price, S, K, r, T, option_type='call',
                           sigma_lo=0.001, sigma_hi=5.0, tol=1e-10, max_iter=200):
    """Compute implied volatility via bisection."""
    price_fn = bsm_call if option_type == 'call' else bsm_put
    
    history = []
    
    for i in range(max_iter):
        sigma_mid = 0.5 * (sigma_lo + sigma_hi)
        price_mid = price_fn(S, K, r, T, sigma_mid)
        error = price_mid - market_price
        history.append((sigma_mid, abs(error)))
        
        if abs(error) < tol or (sigma_hi - sigma_lo) < tol:
            return sigma_mid, i + 1, history
        
        if error > 0:
            sigma_hi = sigma_mid
        else:
            sigma_lo = sigma_mid
    
    return sigma_mid, max_iter, history


# Compare Newton vs Bisection
iv_n, iters_n, hist_n = implied_vol_newton(market_price, S0, K, r, T, 'call', sigma0=0.5)
iv_b, iters_b, hist_b = implied_vol_bisection(market_price, S0, K, r, T, 'call')

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy([h[1] for h in hist_n], 'o-', color=PRIMARY, label=f'Newton-Raphson ({iters_n} iters)')
ax.semilogy([h[1] for h in hist_b], 's-', color=SECONDARY, label=f'Bisection ({iters_b} iters)',
            markersize=3)
ax.set_xlabel('Iteration')
ax.set_ylabel('|Price Error|')
ax.set_title('Convergence: Newton-Raphson vs Bisection')
ax.legend()
plt.tight_layout()
plt.show()

### Interpreting the Convergence Plot

The convergence comparison is dramatic. Newton-Raphson (blue) plummets to machine precision in about 5 iterations. Bisection (orange) decreases steadily but takes 30+ iterations.

On the log-scale y-axis, the key visual signatures are:

- **Newton-Raphson:** The curve bends *downward* more and more steeply --- each step is a bigger drop than the last (in log space). This accelerating descent is the hallmark of quadratic convergence.
- **Bisection:** The curve descends in a straight line (on the log scale) --- each step reduces the error by a constant factor of approximately 2. This steady, predictable descent is linear convergence.

### Cost Comparison

Each Newton-Raphson iteration requires **two function evaluations** (one for price, one for vega), while each bisection iteration requires only **one** (just the price). So the fair comparison is:

- Newton-Raphson: ~5 iterations $\times$ 2 evaluations = ~10 function evaluations
- Bisection: ~35 iterations $\times$ 1 evaluation = ~35 function evaluations

Newton-Raphson is still about 3.5x faster, but the gap is narrower than the raw iteration count suggests.

> **Important:** In production systems, even faster methods exist. Jaeckel (2015) developed a method called "Let's be rational" that computes IV with only two function evaluations, achieving machine precision without iteration. For our educational purposes, Newton-Raphson and bisection are sufficient.

> **Key Concept:** The choice between Newton-Raphson and bisection is a microcosm of a general theme in numerical methods: *speed vs. robustness*. Fast methods exploit structure (derivatives, smoothness) but can fail when that structure breaks down. Slow methods make fewer assumptions and always work. The best practical systems combine both --- using the fast method when it works and falling back to the slow method when it does not.

---

## 5. The Volatility Smile and Skew

### The Puzzle

If Black-Scholes were exactly correct, implied volatility would be the **same for all strikes** at a given expiration. After all, there is only one true volatility $\sigma$, and it should not depend on which option you look at.

In reality, IV varies systematically across strikes. This pattern is called the **volatility smile** or **skew**.

This is one of the most important empirical facts in finance. It tells us that the BSM model is *wrong* --- but wrong in a specific, structured, and informative way.

### What Do We See in Practice?

| Market | Pattern | Shape | Why |
|--------|---------|-------|-----|
| **Equity** | Volatility **skew** | Higher IV at low strikes, lower at high | Demand for crash protection; leverage effect |
| **FX** | Volatility **smile** | U-shaped; higher IV at both extremes | Symmetric tail risk in currency markets |
| **Commodity** | Mixed | Depends on the commodity | Supply shocks (up) and demand crashes (down) |

### What Does the Skew Tell Us About Market Expectations?

The shape of the IV curve across strikes is not arbitrary. It encodes the market's beliefs about the true probability distribution of future stock returns. To understand this, consider what happens when we compute IV for an OTM put with a very low strike:

- BSM assumes log-normal returns, which assign a very small probability to extreme drops.
- If the market price of this OTM put is *higher* than BSM would predict using the ATM volatility, the IV for that put will be higher than the ATM IV.
- This elevated IV is the market saying: "We think a crash is more likely than the log-normal distribution assumes."

The equity skew reveals that the market believes:

1. **Large drops are more likely than BSM predicts.** The lognormal distribution underestimates the probability of crashes. In statistical language, the true return distribution has a **fat left tail**.
2. **There is demand for downside protection.** Portfolio managers buy OTM puts to hedge, driving up their prices (and hence their IV). This is a supply-demand effect layered on top of the distributional effect.
3. **The "leverage effect":** When a stock drops, its debt-to-equity ratio rises, making it riskier, which increases its volatility. This creates a negative correlation between stock returns and volatility.

### Why the Smile Exists: Three Deep Reasons

The volatility smile and skew are not just statistical curiosities --- they reflect fundamental features of financial markets that BSM ignores:

**1. Fat Tails (Real Crashes Happen)**

The BSM model assumes stock returns follow a normal distribution (after taking logs). But real returns have "fat tails" --- extreme moves happen far more often than a normal distribution predicts. The 1987 crash (22% in one day), the 2008 financial crisis, and the 2020 COVID crash were all events with probabilities so small under the normal distribution that they should essentially never occur. Yet they do.

When the market prices options to account for these fat tails, the result is elevated IV for OTM options (both puts and calls), creating the smile.

**2. The Leverage Effect**

When a company's stock price falls, its debt-to-equity ratio mechanically increases (the debt stays the same but the equity is worth less). Higher leverage means higher risk, which means higher volatility. This creates a negative correlation between stock returns and volatility changes.

The leverage effect explains why the equity skew is asymmetric: IV rises more for low strikes (associated with stock price declines and higher leverage) than for high strikes (associated with stock price increases and lower leverage).

**3. Jumps in Asset Prices**

BSM assumes prices move continuously --- no jumps, no gaps. But in reality, stocks can gap down overnight on bad earnings, or gap up on a takeover announcement. These jumps are particularly important for short-dated options and OTM options, and they contribute to elevated IV at extreme strikes.

> **Key Concept:** The volatility smile/skew is the market telling you that BSM is wrong. The pattern of implied volatilities across strikes encodes information about the true (non-lognormal) distribution of stock returns that the market expects. Higher IV at a strike means the market thinks that region is more likely or more feared than BSM would suggest.

### Historical Note: Before and After 1987

Before the 1987 stock market crash, equity options displayed a relatively flat IV across strikes. After "Black Monday" (when the S&P 500 fell 22% in one day), the skew appeared and has persisted ever since. The crash demonstrated that extreme events are far more likely than the lognormal distribution suggests, and the options market permanently adjusted its pricing.

This is one of the most dramatic examples of a market "learning" from a single event. Before 1987, the market priced options as if crashes were nearly impossible. After 1987, the market permanently incorporated crash risk into option prices, and the skew has never disappeared.

> **Important:** The post-1987 skew is often called the "smirk" rather than the "smile" because it is asymmetric --- IV increases much more for low strikes than for high strikes. The symmetric "smile" shape is more typical of FX markets, where currencies can move sharply in either direction.

### Generating and Extracting the Smile

The code below creates synthetic option prices that embed a realistic IV smile/skew, then uses our Newton-Raphson solver to extract the IV curve. This simulates what a practitioner does: observe market prices, extract IV at each strike, and plot the smile.

The parametric model uses log-moneyness $m = \log(K / F)$ where $F = Se^{rT}$ is the forward price:

- **Equity skew:** $\sigma(m) = 0.20 - 0.15m + 0.10m^2$ --- a tilted parabola, steeper on the downside.
- **FX smile:** $\sigma(m) = 0.10 + 0.08m^2$ --- a symmetric parabola.

The fact that our solver perfectly recovers the true IV from the synthetic prices is a validation of both the solver and our understanding of the inversion problem.

In [ ]:
def generate_skew_prices(S0, r, T, strikes, skew_type='equity'):
    """Generate synthetic option prices with volatility smile/skew.
    
    Uses a simple parametric IV model to create 'market' prices.
    """
    moneyness = np.log(strikes / (S0 * np.exp(r * T)))  # log-moneyness
    
    if skew_type == 'equity':
        # Equity skew: higher IV for low strikes
        base_vol = 0.20
        iv_true = base_vol - 0.15 * moneyness + 0.10 * moneyness**2
    elif skew_type == 'fx':
        # FX smile: symmetric U-shape
        base_vol = 0.10
        iv_true = base_vol + 0.08 * moneyness**2
    else:
        iv_true = np.full_like(moneyness, 0.20)
    
    iv_true = np.maximum(iv_true, 0.01)  # floor
    prices = np.array([bsm_call(S0, K, r, T, sig) for K, sig in zip(strikes, iv_true)])
    
    return prices, iv_true


# Extract implied vol from synthetic prices
strikes = np.linspace(70, 130, 50)
T = 0.5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (skew_type, ax) in enumerate(zip(['equity', 'fx'], axes)):
    prices, iv_true = generate_skew_prices(S0, r, T, strikes, skew_type)
    
    # Extract IV using Newton-Raphson
    iv_extracted = []
    for K_i, p_i in zip(strikes, prices):
        iv, _, _ = implied_vol_newton(p_i, S0, K_i, r, T, 'call')
        iv_extracted.append(iv)
    iv_extracted = np.array(iv_extracted)
    
    ax.plot(strikes, iv_true * 100, 'o', color=SECONDARY, markersize=4, label='True IV')
    ax.plot(strikes, iv_extracted * 100, '-', color=PRIMARY, linewidth=2, label='Extracted IV')
    ax.axvline(S0, color='gray', linestyle='--', alpha=0.5, label=f'S = {S0}')
    ax.set_xlabel('Strike K')
    ax.set_ylabel('Implied Volatility (%)')
    ax.set_title(f'{skew_type.upper()} Volatility {"Skew" if skew_type == "equity" else "Smile"}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

### Reading the Smile Plots

The left plot shows the equity **skew**: IV is highest for low strikes (OTM puts, reflecting crash fear) and decreases for high strikes. Notice the characteristic shape --- a steep downward slope on the left, with a gentle flattening on the right. This is the "smirk" that dominates equity markets.

The right plot shows the FX **smile**: IV is lowest at the money and increases symmetrically for both OTM puts and calls. This U-shape reflects the fact that currencies can move sharply in either direction --- a dollar can strengthen or weaken against the euro with roughly equal probability of extreme moves.

Our Newton-Raphson solver perfectly recovers the true IV from the synthetic prices, confirming the implementation works correctly.

### Practical Trading Implications

The smile has direct consequences for option traders:

- **OTM puts are "expensive"** (high IV) relative to ATM options. If you think crash risk is overstated, you might sell OTM puts --- but this is the trade that blew up many hedge funds in 2008.
- **Skew can be traded** using risk reversals (buy a call, sell a put at the same delta) or butterfly spreads (buy strikes on either side of ATM, sell ATM). These strategies have payoffs that depend on the *shape* of the smile rather than the *level* of volatility.
- **The skew changes over time.** Before earnings announcements, the smile might steepen (more crash fear). After the announcement resolves uncertainty, the smile might flatten.

> **Common Mistake:** Assuming the smile is static. In reality, the entire smile shifts and changes shape constantly. A position that looks profitable under today's smile might lose money if the smile changes. This is called **vega risk** (exposure to parallel shifts in the smile) and **skew risk** (exposure to changes in the smile's shape).

---

## 6. The Volatility Surface

### From Smile to Surface

The **volatility surface** $\sigma_{\text{imp}}(K, T)$ extends the smile concept to two dimensions: IV as a function of both strike and maturity. It captures the complete structure of option prices across the market.

If the smile answers "how does IV vary across strikes at a fixed maturity?", the surface answers the more complete question: "how does IV vary across *all* strikes and *all* maturities simultaneously?"

### Key Features of the Surface

1. **Strike dimension (the smile/skew):** Already discussed. The skew is steeper for short maturities and flattens for long maturities.

2. **Maturity dimension (term structure):** ATM IV can be upward sloping (normal markets), inverted (high-volatility regimes), or humped.

3. **Skew flattening:** Short-term options have a steep skew because a single large move can drastically change moneyness. Long-term options have a flatter skew because many moves average out.

### Why the Skew Flattens with Maturity

This is an important and often-misunderstood point. Consider a 1-week option versus a 1-year option:

- **1-week option:** A single bad day can easily move the stock 5--10%, turning an ATM option deep ITM or OTM. One jump event can dominate the outcome. Since jumps and crashes are the main driver of the skew, the skew is steepest for short maturities.
- **1-year option:** Over 252 trading days, the law of large numbers kicks in. Individual jumps are "averaged out" by the many subsequent days of normal returns. The distribution of the final stock price is closer to log-normal (by the central limit theorem), so the BSM assumption is more accurate, and the skew is milder.

This is why the volatility surface typically has a "ski slope" shape: steep at the front (short maturities), flattening toward the back (long maturities).

> **Key Concept:** The volatility surface is the market's complete description of option prices. It is the single most important object in derivatives trading. Any option pricing or hedging decision ultimately comes down to understanding this surface and how it changes over time.

> **Important:** The surface must satisfy no-arbitrage constraints. For instance, you cannot have two adjacent strikes where the butterfly spread (buying the wings, selling the body) has a negative price. Similarly, calendar spreads (buying a longer-dated option, selling a shorter-dated one at the same strike) must have non-negative value. These constraints restrict the set of valid surfaces and make interpolation a non-trivial mathematical problem.

### Building and Visualizing the Surface

The code below generates a synthetic volatility surface with the key empirical features: skew that flattens with maturity, and a term structure of base volatility. We visualize it as both a 3D surface plot and a heatmap.

The parametric model decomposes IV into three components:

$$\sigma_{\text{imp}}(K, T) = \underbrace{\sigma_{\text{base}}(T)}_{\text{term structure}} + \underbrace{\beta(T) \cdot m}_{\text{skew}} + \underbrace{\gamma(T) \cdot m^2}_{\text{curvature}}$$

where $m = \log(K / F)$ is log-moneyness, $\beta(T) \propto 1/\sqrt{T}$ (skew decays with maturity), and $\gamma(T) \propto 1/T$ (curvature also decays).

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

def generate_vol_surface(S0, r, strikes, maturities):
    """Generate a synthetic volatility surface with realistic features."""
    KK, TT = np.meshgrid(strikes, maturities)
    
    moneyness = np.log(KK / (S0 * np.exp(r * TT)))
    
    # Parametric model: skew flattens with maturity, base vol has term structure
    base_vol = 0.18 + 0.05 * np.exp(-2 * TT)  # term structure
    skew = -0.12 / np.sqrt(TT + 0.1)  # skew decreases with T
    curvature = 0.08 / (TT + 0.1)
    
    iv_surface = base_vol + skew * moneyness + curvature * moneyness**2
    iv_surface = np.maximum(iv_surface, 0.02)
    
    return iv_surface, KK, TT


strikes = np.linspace(70, 130, 40)
maturities = np.linspace(0.1, 2.0, 40)

iv_surf, KK, TT = generate_vol_surface(S0, r, strikes, maturities)

# 3D surface plot
fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(KK, TT, iv_surf * 100, cmap='viridis', alpha=0.8, edgecolor='none')
ax1.set_xlabel('Strike K')
ax1.set_ylabel('Maturity T')
ax1.set_zlabel('IV (%)')
ax1.set_title('Volatility Surface')
ax1.view_init(elev=25, azim=-60)

# Heatmap
ax2 = fig.add_subplot(122)
im = ax2.pcolormesh(KK, TT, iv_surf * 100, cmap='viridis', shading='auto')
ax2.set_xlabel('Strike K')
ax2.set_ylabel('Maturity T')
ax2.set_title('Volatility Surface (Heatmap)')
plt.colorbar(im, ax=ax2, label='IV (%)')

plt.tight_layout()
plt.show()

### Reading the Surface Visualizations

The 3D surface and heatmap show the key features:

- **Skew at short maturities** (front of the surface): steep slope from high IV at low strikes to low IV at high strikes. The front-left corner (low strike, short maturity) has the highest IV, reflecting the market's intense fear of short-term crashes.
- **Flattening with maturity** (back of the surface): the skew gradually disappears for longer-dated options. The surface becomes flatter along the strike axis as maturity increases.
- **Term structure**: the overall level of IV varies with maturity. In our model, short-term base vol is slightly elevated (the $0.05 e^{-2T}$ term), which produces a mildly inverted term structure.

On the heatmap, look for the "hot" region in the bottom-left (low strike, short maturity) and the "cool" region in the top-right (high strike, long maturity). The color gradient tells you how IV varies across the two dimensions simultaneously.

> **Key Concept:** Every option that trades in the market corresponds to a single point on this surface. When a trader says they are "buying the surface" at a particular (K, T) point, they mean they are buying an option whose IV can be read off this surface. The entire business of exotic options and structured products involves understanding how the surface behaves and how it will change.

> **Important:** In practice, the volatility surface is interpolated from a discrete set of observed option prices. The surface must be free of arbitrage (no butterfly spread arbitrage, no calendar spread arbitrage), which imposes constraints on the interpolation. This is a non-trivial problem that occupies much of quantitative finance. Common interpolation methods include SABR, SVI (stochastic volatility inspired), and local volatility approaches.

---

## 7. Term Structure of Volatility

### What Is the Term Structure?

The **term structure** is the ATM implied volatility as a function of maturity $T$. It reveals the market's expectations about how volatility will evolve over time.

While the smile/skew tells us about the market's beliefs about the *distribution* of returns at a fixed maturity, the term structure tells us about the market's beliefs about the *dynamics* of volatility itself.

### Three Common Shapes

| Shape | When It Appears | What It Means |
|-------|----------------|---------------|
| **Upward sloping** | Calm markets | Market expects volatility to increase from current low levels |
| **Inverted** | After a crash or high-vol event | Market expects current high volatility to decrease (mean reversion) |
| **Flat** | Normal conditions | Market expects roughly constant volatility |

### The Role of Mean Reversion

Volatility tends to **mean-revert**: extreme levels (both high and low) tend to return to a long-run average. This is one of the most robust empirical facts in finance, and it fundamentally shapes the term structure.

Mean reversion means:
- After a crash (high vol), short-term IV is high but long-term IV is lower --- the market expects things to calm down.
- In calm periods (low vol), long-term IV is higher than short-term --- the market knows calm cannot last forever.

To see why, consider an analogy with temperature. If today is an unusually hot 40C day, you would not expect tomorrow, or the day after, to also be 40C. You would expect temperatures to gradually return to the seasonal average. Similarly, if volatility is unusually high today (say VIX = 50), the market does not expect it to stay at 50 for the next two years. It expects a gradual return to the long-run average of around 15--20.

This mean-reverting expectation is directly encoded in the term structure:

- **Short-term IV** reflects *current* volatility conditions.
- **Long-term IV** reflects the *long-run average* volatility.
- **The slope** of the term structure tells you how fast the market expects volatility to revert to the mean.

> **Key Concept:** The term structure of IV encodes the market's expectation of how volatility will evolve. If short-term IV is much higher than long-term IV (inverted term structure), the market is saying: "Things are scary right now, but we expect them to normalize." If short-term IV is much lower than long-term IV (steep upward slope), the market is saying: "Things are unusually calm, and we expect more normal (higher) volatility in the future." This information is valuable for trading and risk management.

### Connection to Volatility Models

The term structure is closely related to stochastic volatility models. In the Heston model, for example, volatility follows a mean-reverting square-root process:

$$dv_t = \kappa(\theta - v_t)\,dt + \xi \sqrt{v_t}\,dW_t$$

where $v_t = \sigma_t^2$ is the instantaneous variance, $\theta$ is the long-run mean, $\kappa$ is the speed of mean reversion, and $\xi$ is the "vol of vol." The expected variance over a horizon $T$ is:

$$\mathbb{E}[\bar{v}_T] = \theta + (v_0 - \theta)\frac{1 - e^{-\kappa T}}{\kappa T}$$

This formula shows that the expected average variance interpolates between the current variance $v_0$ (for short $T$) and the long-run mean $\theta$ (for long $T$), with the interpolation speed controlled by $\kappa$. This is exactly the shape we see in the empirical term structure.

> **Common Mistake:** Interpreting the term structure as a forecast of realized volatility at specific future dates. The term structure tells you what the market expects the *average* volatility to be *over the life of the option*, not what the *instantaneous* volatility will be at expiry. These are different quantities, and confusing them leads to incorrect trading decisions.

### Visualizing the Term Structure Across Market Regimes

The code below generates ATM term structures under three market regimes: normal, stressed (post-crash), and calm. Watch how the shape reflects the market's expectations about volatility mean reversion.

In [ ]:
# ATM term structure
maturities_ts = np.linspace(0.05, 3.0, 100)
K_atm = S0  # ATM strike

# Generate different regimes
def atm_term_structure(T, regime='normal'):
    if regime == 'normal':
        return 0.18 + 0.03 * (1 - np.exp(-T))
    elif regime == 'stressed':
        return 0.35 * np.exp(-0.5 * T) + 0.18 * (1 - np.exp(-0.5 * T))
    elif regime == 'calm':
        return 0.12 + 0.06 * (1 - np.exp(-0.8 * T))

fig, ax = plt.subplots(figsize=(10, 6))
for regime, color, label in [('normal', PRIMARY, 'Normal'), ('stressed', SECONDARY, 'Stressed'),
                              ('calm', TERTIARY, 'Calm')]:
    iv_ts = atm_term_structure(maturities_ts, regime)
    ax.plot(maturities_ts, iv_ts * 100, color=color, linewidth=2, label=label)

ax.set_xlabel('Maturity T (years)')
ax.set_ylabel('ATM Implied Volatility (%)')
ax.set_title('Term Structure of ATM Volatility')
ax.legend()
plt.tight_layout()
plt.show()

### Interpreting the Three Regimes

The three curves show typical term structure shapes:

- **Normal** (blue): gently upward sloping, with IV rising from ~18% to ~21% as maturity increases. This is the "default" shape in most markets --- the market expects a modest increase in volatility over time, reflecting the general uncertainty about the future.

- **Stressed** (orange): sharply inverted, starting at 35% for short maturities and declining toward 18% for longer ones. This is what you see after a market crash. The short end is elevated because *right now* the market is turbulent, but the long end is much lower because the market expects the storm to pass. The rate at which the curve flattens tells you how quickly the market expects normalization.

- **Calm** (green): steeply upward sloping, with IV starting at only 12% and rising as the market prices in future uncertainty. This is the calm-before-the-storm shape. The market knows that current tranquility is unusual and expects volatility to increase toward its long-run mean.

Notice that all three curves converge toward a similar long-term level (around 18--21%). This is the market's estimate of the **long-run average volatility** --- the $\theta$ parameter in the Heston model. Regardless of whether the market is currently stressed or calm, the long-run expectation is similar.

> **Key Concept:** The convergence of all term structure shapes toward a common long-run level is direct evidence of mean reversion in volatility. If volatility were a random walk (no mean reversion), the term structure would be flat regardless of current conditions. The fact that it tilts up in calm markets and inverts in stressed markets is the market's way of saying: "Volatility reverts to a mean, and we know what that mean is."

> **Important:** The speed of convergence (how quickly the curve flattens) corresponds to the mean-reversion speed $\kappa$. Fast convergence means the market expects volatility to normalize quickly. Slow convergence means the market thinks the current regime will persist for a while. This has direct implications for the relative pricing of short-dated versus long-dated options.

---

## 8. The VIX Index: Wall Street's Fear Gauge

### What Is the VIX?

The **CBOE Volatility Index (VIX)** is the market's expectation of 30-day realized volatility, derived from S&P 500 option prices. It is often called the "fear index" because it spikes during market crises.

The VIX distills the entire S&P 500 options market --- thousands of individual option prices across dozens of strikes and two expirations --- into a single number that captures the market's overall level of fear.

### Why the VIX Matters

The VIX is important for several reasons:

1. **Sentiment indicator:** A rising VIX signals increasing market anxiety. Portfolio managers use it as a real-time measure of market stress.
2. **Risk management:** Many risk models use the VIX to adjust position sizes --- when VIX is high, positions are reduced; when VIX is low, positions can be increased.
3. **Tradable asset class:** VIX futures, options on VIX, and VIX-linked ETFs form an entire ecosystem of volatility-related products.
4. **Macro indicator:** Central banks and policymakers monitor the VIX as a measure of financial system stress.

### Historical VIX Levels

| Level | Interpretation | Example Period |
|-------|---------------|----------------|
| 10--15 | Low fear, complacent market | 2017 bull market |
| 15--20 | Normal, healthy uncertainty | Typical conditions |
| 20--30 | Elevated concern | Trade wars, mild recessions |
| 30--50 | High fear, significant uncertainty | COVID crash (Mar 2020) |
| 50--80+ | Extreme panic | 2008 financial crisis |

The long-run average VIX is approximately 19--20. It spends most of its time between 12 and 25, with occasional spikes above 40 during crises. The VIX has only exceeded 80 twice in its history: during the 2008 financial crisis (peak ~80) and briefly during the COVID crash.

### A Key Asymmetry: The VIX as a Skewed Gauge

The VIX has a pronounced negative correlation with the stock market: when the S&P 500 drops sharply, the VIX spikes. But the relationship is asymmetric:

- A 5% market drop might cause the VIX to spike from 15 to 30 (a 100% increase).
- A 5% market rally might only cause the VIX to drop from 15 to 12 (a 20% decrease).

This asymmetry reflects the fundamental nature of fear: it arrives suddenly and dissipates slowly. Markets take the stairs up and the elevator down.

### The Conceptual Connection: VIX and Implied Volatility

The VIX is essentially a *weighted average* of implied volatilities from S&P 500 options. But it is not a simple average. The weighting scheme, based on the variance swap replication formula, gives:

- **More weight to OTM options** (both puts and calls)
- **Even more weight to low-strike puts** (because of the $1/K^2$ weighting)

This means the VIX is particularly sensitive to the price of crash protection. When investors bid up OTM put prices (because they fear a crash), the VIX rises even if ATM volatility is unchanged.

> **Key Concept:** The VIX bridges the gap between individual option prices and a single measure of market fear. It aggregates information from across the entire option chain, weighting each option by its contribution to the variance of the S&P 500. Because OTM puts (crash insurance) are weighted heavily, the VIX captures downside fear more than upside uncertainty.

### How the VIX Is Computed

The VIX is based on the model-free implied variance:

$$\sigma^2_{\text{VIX}} = \frac{2}{T} \sum_i \frac{\Delta K_i}{K_i^2}\,e^{rT}\,Q(K_i)$$

where $Q(K_i)$ is the midpoint price of the OTM option at strike $K_i$ (put if $K_i < F$, call if $K_i > F$, average at $K_i = F$). Then $\text{VIX} = 100 \times \sigma_{\text{VIX}}$.

Let us unpack this formula:

- **$\frac{2}{T}$:** Annualizes the variance (the VIX is quoted in annualized volatility units).
- **$\frac{\Delta K_i}{K_i^2}$:** The weighting factor. The $1/K_i^2$ gives more weight to lower strikes (OTM puts), reflecting their outsized contribution to variance. The $\Delta K_i$ accounts for the spacing between available strikes.
- **$e^{rT}$:** Adjusts option prices from present value to forward value.
- **$Q(K_i)$:** Uses only OTM options because they are more liquid and have cleaner prices. OTM puts capture downside risk; OTM calls capture upside risk.

The beauty of this formula is that it is **model-free** --- it does not assume BSM, Heston, or any other model. It directly extracts the market's implied variance from option prices. This makes it a robust measure of market fear.

> **Important:** You cannot directly invest in the VIX. VIX futures and VIX ETFs (like VXX) are related but not identical to the VIX index itself. The difference between VIX and VIX futures is called the "term structure of VIX" and is itself a tradable signal. Typically, VIX futures trade above the VIX spot (called "contango"), meaning investors pay a premium for future volatility protection.

> **Common Mistake:** Thinking VIX = 20 means the market expects a 20% move in the S&P 500. It means the market expects the *annualized standard deviation* of returns to be 20%. Over 30 days, this translates to an expected move of $20\% \times \sqrt{30/365} \approx 5.7\%$. To get the expected monthly move, divide the VIX by $\sqrt{12} \approx 3.46$.

### Simplified VIX Implementation

The code below implements a simplified version of the VIX calculation. We generate synthetic S&P 500 option prices with different base volatility levels and compute the resulting VIX-like index.

Key details in the implementation:
- We use 30-day options ($T = 30/365$), matching the VIX's target maturity.
- The forward price $F = Se^{rT}$ determines the boundary between using OTM puts and OTM calls.
- A forward correction term $(F/K_0 - 1)^2 / T$ removes a small bias.

We test under three volatility regimes to see how the VIX tracks the overall level of option-implied uncertainty.

In [ ]:
def simplified_vix(S0, r, T, strikes, option_prices_call, option_prices_put):
    """Simplified VIX-style calculation from option prices.
    
    Uses model-free implied variance based on out-of-the-money option prices.
    """
    F = S0 * np.exp(r * T)  # forward price
    
    # Use OTM options: puts below forward, calls above
    dK = np.diff(strikes)
    dK = np.append(dK, dK[-1])  # pad
    
    variance = 0.0
    for i, K in enumerate(strikes):
        if K < F:
            Q = option_prices_put[i]
        elif K > F:
            Q = option_prices_call[i]
        else:
            Q = 0.5 * (option_prices_call[i] + option_prices_put[i])
        
        variance += (dK[i] / K**2) * np.exp(r * T) * Q
    
    variance *= 2.0 / T
    
    # Subtract forward correction
    K0 = strikes[np.argmin(np.abs(strikes - F))]  # strike closest to F
    variance -= (1.0 / T) * (F / K0 - 1)**2
    
    vix = 100 * np.sqrt(max(variance, 0))
    return vix, variance


# Compute simplified VIX
T_vix = 30 / 365  # 30 days
strikes_vix = np.arange(70, 131, 1.0)

# Generate prices with different vol levels
for base_sigma, label in [(0.15, 'Low Vol'), (0.20, 'Normal'), (0.35, 'High Vol')]:
    moneyness = np.log(strikes_vix / (S0 * np.exp(r * T_vix)))
    iv_strikes = base_sigma - 0.1 * moneyness + 0.05 * moneyness**2
    iv_strikes = np.maximum(iv_strikes, 0.01)
    
    calls = np.array([bsm_call(S0, K, r, T_vix, sig) for K, sig in zip(strikes_vix, iv_strikes)])
    puts = np.array([bsm_put(S0, K, r, T_vix, sig) for K, sig in zip(strikes_vix, iv_strikes)])
    
    vix, var = simplified_vix(S0, r, T_vix, strikes_vix, calls, puts)
    print(f"{label:12s}: VIX = {vix:.2f}, ATM IV = {base_sigma*100:.1f}%")

### Interpreting the VIX Output

The VIX tracks the ATM IV closely but is not identical, because it incorporates information from options across all strikes (weighted by $1/K^2$, which gives more weight to lower strikes --- reflecting the skew).

You should notice that the VIX is typically slightly *higher* than the ATM IV. This is because the skew adds extra variance from the OTM put side: the elevated IV at low strikes contributes additional variance to the VIX calculation that is not reflected in the ATM IV alone.

This difference between VIX and ATM IV is itself informative:
- When VIX is much higher than ATM IV, it means the skew is steep --- the market is particularly worried about crash risk.
- When VIX is close to ATM IV, the skew is flat --- the market sees roughly symmetric risk.

> **Key Concept:** The VIX is essentially a weighted average of out-of-the-money option implied volatilities. Because OTM puts (low strikes) are weighted more heavily, the VIX captures crash risk more than upside risk. This is why the VIX tends to spike more during crashes than during rallies of equal magnitude.

> **Important:** Our simplified VIX captures the essential mechanics, but the real VIX calculation includes additional details: interpolation between two expiration months to target exactly 30 days, exclusion of options with zero bid prices, and specific rules for determining the forward price. These details matter for exact replication but do not change the conceptual framework.

---

## 9. Summary: The Big Picture

Let us step back and see how all the pieces fit together:

1. **Implied volatility** is the market's way of expressing option prices in a standardized, comparable format. It is the $\sigma$ that makes BSM match the market price.

2. **Computing IV** requires solving a nonlinear equation. Newton-Raphson is fast (quadratic convergence via vega), and bisection is robust (guaranteed linear convergence). Production systems use hybrids.

3. **The volatility smile/skew** reveals that BSM is wrong --- the market knows returns have fat tails, crashes are more likely than the normal distribution suggests, and volatility itself is stochastic.

4. **The volatility surface** $\sigma_{\text{imp}}(K, T)$ is the complete description of option prices. Its shape encodes the market's beliefs about the distribution of returns (via the smile) and the dynamics of volatility (via the term structure).

5. **The VIX** aggregates the entire surface into a single number, providing a real-time measure of market fear.

The deeper lesson is that implied volatility is far more than a technical calculation. It is a *language* --- the language in which the options market communicates its beliefs about risk, uncertainty, and the future. Learning to read and interpret implied volatility is learning to understand what millions of market participants collectively think about the world.

---

## 10. References

1. Gatheral, J. (2006). *The Volatility Surface: A Practitioner's Guide*. Wiley.
2. Dupire, B. (1994). *Pricing with a smile*. Risk, 7(1), 18-20.
3. Hull, J. C. (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson.
4. CBOE (2019). *VIX White Paper*. Chicago Board Options Exchange.
5. Carr, P., & Madan, D. (1998). *Towards a theory of volatility trading*. Volatility: New Estimation Techniques for Pricing Derivatives, 29, 417-427.
6. Jaeckel, P. (2015). *Let's be rational*. Wilmott, 2015(75), 40-53.